In [1]:
import pandas as pd
import numpy as np
import math

In [2]:
def compute_weighted_metrics(N, n, y1, threshold):
    """
    Parameters:
        N : list of total number of reports per bin 
        n : list of sampled reports per bin 
        y1 : list of number of Y=1 in each sampled bin 
        threshold : prediction threshold (e.g., 0.4)
    
    Returns:
        precision, recall, accuracy
    """
    assert len(N) == len(n) == len(y1), "Input lists must have same length"
    size = len(N)

    # Step 1: Compute weights for each bin
    weight = [N[i] / n[i] if n[i] != 0 else 0 for i in range(size)]

    # Step 2: Initialize accumulators
    weighted_TP = 0
    precision_denom = 0
    recall_denom = 0
    weighted_TN = 0
    total_reports = 0

    # Step 3: Determine threshold bin index
    bin_edges = [i / 10 for i in range(11)]  # [0.0, 0.1, ..., 1.0]
    threshold_bin_index = next(i for i, edge in enumerate(bin_edges) if edge >= threshold)

    # Step 4: Loop over bins
    for i in range(size):
        w = weight[i]
        if w == 0:
            continue  # skip bins with zero samples

        if i >= threshold_bin_index:
            # Bin predicts Ŷ = 1
            weighted_TP += w * y1[i]
            precision_denom += N[i]
        else:
            # Bin predicts Ŷ = 0
            y0 = n[i] - y1[i]
            weighted_TN += w * y0

        # Always accumulate for recall and total count
        recall_denom += w * y1[i]
        total_reports += N[i]

    # Step 5: Compute metrics
    if precision_denom > 0:
        precision = weighted_TP / precision_denom
    else:
        precision = float('nan')

    if recall_denom > 0:
        recall = weighted_TP / recall_denom
    else:
        recall = float('nan')

    if total_reports > 0:
        accuracy = (weighted_TP + weighted_TN) / total_reports
    else:
        accuracy = float('nan')

    return precision, recall, accuracy


In [3]:
# ON table
N = [3518.1, 32.23, 26.65, 16.9, 10, 11, 275.478, 12.03, 26.31, 1945.464]
n = [59, 19, 18, 13, 10, 11, 27, 11, 25, 366]
y1 = [5, 7, 6, 8, 4, 3, 6, 3, 8, 302]
precision, recall, accuracy = compute_weighted_metrics(N, n, y1, 0.4)
print(precision, recall, accuracy)

0.7390274607240395 0.8365337221374946 0.8426341330965123


In [4]:
# ON table (threshold 0.5)
N = [3518.1, 32.23, 26.65, 16.9, 10, 11, 275.478, 12.03, 26.31, 1945.464]
n = [59, 19, 18, 13, 10, 11, 27, 11, 25, 366]
y1 = [5, 7, 6, 8, 4, 3, 6, 3, 8, 302]
precision, recall, accuracy = compute_weighted_metrics(N, n, y1, 0.5)
f1  = (2 * precision * recall / (precision + recall)
        if (not math.isnan(precision) and not math.isnan(recall) and (precision + recall) > 0)
        else float('nan'))
print(precision, recall, accuracy, f1)

0.7405207882521793 0.8345481105027341 0.8429746071930728 0.7847278618889352


In [5]:
print(2 * precision * recall / (precision + recall))

0.7847278618889352


In [6]:
# BC 0.5
# ON table (threshold 0.5)
N = [726.24, 8, 5, 1, 1, 1, 2, 0, 3, 229]
n = [178, 8, 5, 1, 1, 1, 2, 0, 3, 229]
y1 = [16, 4, 5, 0, 0, 0, 0, 0, 1, 196]
precision, recall, accuracy = compute_weighted_metrics(N, n, y1, 0.5)
f1  = (2 * precision * recall / (precision + recall)
        if (not math.isnan(precision) and not math.isnan(recall) and (precision + recall) > 0)
        else float('nan'))
print('Precision: ', precision)
print('Recall:', recall)
print('Acc: ', accuracy)
print("F1", 2 * precision * recall / (precision + recall))

Precision:  0.8382978723404255
Recall: 0.7261869654969036
Acc:  0.8849872982053594
F1 0.7782254878723237


In [7]:
# BC 0.4
# ON table (threshold 0.5)
N = [726.24, 8, 5, 1, 1, 1, 2, 0, 3, 229]
n = [178, 8, 5, 1, 1, 1, 2, 0, 3, 229]
y1 = [16, 4, 5, 0, 0, 0, 0, 0, 1, 196]
precision, recall, accuracy = compute_weighted_metrics(N, n, y1, 0.4)
f1  = (2 * precision * recall / (precision + recall)
        if (not math.isnan(precision) and not math.isnan(recall) and (precision + recall) > 0)
        else float('nan'))
print('Precision: ', precision)
print('Recall:', recall)
print('Acc: ', accuracy)
print("F1", 2 * precision * recall / (precision + recall))

Precision:  0.8347457627118644
Recall: 0.7261869654969036
Acc:  0.8839629599278866
F1 0.7766913736003784


In [29]:
# BC 0.3
N = [726.24, 8, 5, 1, 1, 1, 2, 0, 3, 229]
n = [178, 8, 5, 1, 1, 1, 2, 0, 3, 229]
y1 = [16, 4, 5, 0, 0, 0, 0, 0, 1, 196]
precision, recall, accuracy = compute_weighted_metrics(N, n, y1, 0.01)
f1  = (2 * precision * recall / (precision + recall)
        if (not math.isnan(precision) and not math.isnan(recall) and (precision + recall) > 0)
        else float('nan'))
print('Precision: ', precision)
print('Recall:', recall)
print('Acc: ', accuracy)
print("F1", 2 * precision * recall / (precision + recall))

Precision:  0.824
Recall: 0.7593630197581835
Acc:  0.8880603130377777
F1 0.7903621853898097


In [8]:
# AB table (incorrect)
N = [26445, 289, 182, 206, 817, 687, 79, 69, 78, 1148]
n = [1234, 11, 4, 4, 3, 0, 2, 12, 9, 306]
y1 = [28, 2, 0, 0, 1, 0, 0, 4, 2, 239]
precision, recall, accuracy = compute_weighted_metrics(N, n, y1, 0.4)
print(precision, recall, accuracy)
print(2 * precision * recall / (precision + recall))

0.5519430349349538 0.6495012444163959 0.9442470273472773
0.5967612384500419


In [9]:
# first+second round ON calculation
# ON table
N = [4902.53, 39.11, 36, 18.2, 11, 13, 9.3, 14, 29, 1968.46]
n = [89, 25, 27, 14, 11, 13, 18, 14, 29, 374]
y1 = [5, 7, 15, 10, 5, 6, 6, 3, 9, 311]
precision, recall, accuracy = compute_weighted_metrics(N, n, y1, 0.4)
print('Precision: ', precision)
print('Recall:', recall)
print('Acc: ', accuracy)
print("F1", 2 * precision * recall / (precision + recall))

Precision:  0.8132859073820936
Recall: 0.8388911627842861
Acc:  0.900411990209747
F1 0.8258901213913251


In [10]:
print(sum(list([89, 25, 27, 14, 11, 13, 18, 14, 29, 374])))

614


In [11]:
def compute_weighted_metrics_by_report_bins(N, n, y1, yhat1, tp):
    """
    Weighted metrics over bins defined by #reports per patient (not probabilities).

    Inputs are per-bin lists/arrays of equal length:
        N      : total number of reports in the bin
        n      : sampled reports in the bin
        y1     : # human labels Y=1 in the sampled subset
        yhat1  : # model predicted=1 in the sampled subset
        tp     : # (Y=1 AND predicted=1) in the sampled subset

    Returns:
        precision, recall, f1, accuracy
    """
    import math

    # basic checks
    L = len(N)
    assert all(len(x) == L for x in [n, y1, yhat1, tp]), "All inputs must have same length"

    # per-bin weights to scale sample to population
    weight = [ (N[i] / n[i]) if n[i] > 0 else 0.0 for i in range(L) ]

    # derive per-bin sampled confusion pieces
    fp = [ max(yhat1[i] - tp[i], 0) for i in range(L) ]
    fn = [ max(y1[i]    - tp[i], 0) for i in range(L) ]
    tn = [ max(n[i] - (tp[i] + fp[i] + fn[i]), 0) for i in range(L) ]

    # weighted (population-estimated) confusion totals
    TPw = sum(weight[i] * tp[i] for i in range(L))
    FPw = sum(weight[i] * fp[i] for i in range(L))
    FNw = sum(weight[i] * fn[i] for i in range(L))
    TNw = sum(weight[i] * tn[i] for i in range(L))

    total_reports = sum(N)

    # metrics
    precision = TPw / (TPw + FPw) if (TPw + FPw) > 0 else float('nan')
    recall    = TPw / (TPw + FNw) if (TPw + FNw) > 0 else float('nan')
    accuracy  = (TPw + TNw) / total_reports if total_reports > 0 else float('nan')
    f1        = (2 * precision * recall / (precision + recall)
                 if (not math.isnan(precision) and not math.isnan(recall) and (precision + recall) > 0)
                 else float('nan'))

    return precision, recall, f1, accuracy


In [12]:
# AB (weight by patient max threshold 0.4)
# N = [3061, 1462, 1142, 584, 145, 42, 6]
N = [3061, 1588, 1256, 667, 184, 49, 11]
n = [66, 75, 59, 50, 40, 22, 9]
y1 = [1, 3, 4, 3, 7, 8, 3]
yhat1 = [1, 8, 8, 11, 16, 14, 8]
tp = [1, 3, 4, 3, 7, 8, 3]
precision, recall, f1, accuracy = compute_weighted_metrics_by_report_bins(N, n, y1, yhat1, tp)
print("Precision: {}, Recall: {}, F1: {}, Acc: {}".format(precision, recall, f1, accuracy))

print("Weights: ", [N[i]/ n[i] for i in range(len(N))])

Precision: 0.44604494865158373, Recall: 1.0, F1: 0.6169171284302253, Acc: 0.9473864500419167
Weights:  [46.378787878787875, 21.173333333333332, 21.28813559322034, 13.34, 4.6, 2.227272727272727, 1.2222222222222223]


In [13]:
# AB (weight by patient avg threshold 0.5)
# N = [3061, 1462, 1142, 584, 145, 42, 6]
# N = [2933, 1573, 1251, 667, 184, 49, 11]
N = [3061, 1588, 1256, 667, 184, 49, 11]
n = [66, 75, 59, 50, 40, 22, 9]
y1 = [1, 3, 4, 3, 7, 8, 3]
yhat1 = [1, 3, 5, 5, 13, 10, 3]
tp = [1, 3, 4, 3, 7, 8, 3]
precision, recall, f1, accuracy = compute_weighted_metrics_by_report_bins(N, n, y1, yhat1, tp)
print("Precision: {}, Recall: {}, F1: {}, Acc: {}".format(precision, recall, f1, accuracy))

print("Weights: ", [N[i]/ n[i] for i in range(len(N))])

Precision: 0.7830063222860039, Recall: 1.0, F1: 0.8782989858186329, Acc: 0.9882595831796116
Weights:  [46.378787878787875, 21.173333333333332, 21.28813559322034, 13.34, 4.6, 2.227272727272727, 1.2222222222222223]


In [14]:
# AB (weight by report max threshold 0.5)
# N = [3061, 3050, 4069, 3827, 1893, 907, 280]
N = [3061, 3176, 4239, 3993, 1980, 930, 291]
# N = [2933, 2969, 3931, 3732, 1856, 882, 276]
# actual n used for test in correct bins
# n = [66, 129, 171, 253, 375, 382, 209]
n = [97, 154, 193, 293, 429, 431, 236]
# originally #sampled reprots
# n = [100, 160, 200, 302, 438, 439, 239]
y1 = [1, 7, 9, 18, 57, 134, 80]
# yhat1 = [1, 10, 12, 22, 72, 138, 77]
yhat1 = [6, 17, 19, 33, 89, 148, 83]
# tp = [1, 4, 8, 11, 43, 108, 71]
tp = [1, 6, 9, 15, 49, 115, 74]
precision, recall, f1, accuracy = compute_weighted_metrics_by_report_bins(N, n, y1, yhat1, tp)
print("Precision: {}, Recall: {}, F1: {}, Acc: {}".format(precision, recall, f1, accuracy))

print("Weights: ", [N[i]/ n[i] for i in range(len(N))])

Precision: 0.5014362428963912, Recall: 0.8843668286630054, F1: 0.6399950887797801, Acc: 0.9285043891211289
Weights:  [31.556701030927837, 20.623376623376622, 21.963730569948186, 13.627986348122867, 4.615384615384615, 2.1577726218097446, 1.2330508474576272]


In [ ]:
# AB (weight by report max threshold 0.4 impressions)
N = [3061, 3176, 4239, 3993, 1980, 930, 291]
n = [97, 154, 193, 293, 429, 431, 236]
y1 = [1, 7, 9, 21, 57, 144, 80]
yhat1 = [1, 10, 11, 25, 61, 123, 62]
tp = [1, 5, 8, 14, 40, 108, 57]
precision, recall, f1, accuracy = compute_weighted_metrics_by_report_bins(N, n, y1, yhat1, tp)
print("Precision: {}, Recall: {}, F1: {}, Acc: {}".format(precision, recall, f1, accuracy))

print("Weights: ", [N[i]/ n[i] for i in range(len(N))])

Precision: 0.6852264447998446, Recall: 0.7424543784906394, F1: 0.7126934338540261, Acc: 0.9548682085039407
Weights:  [31.556701030927837, 20.623376623376622, 21.963730569948186, 13.627986348122867, 4.615384615384615, 2.1577726218097446, 1.2330508474576272]


In [ ]:
# AB (weight by report max threshold 0.4 emsemble)
N = [3061, 3176, 4239, 3993, 1980, 930, 291]
n = [97, 154, 193, 293, 429, 431, 236]
y1 = [1, 7, 9, 21, 57, 144, 80]
yhat1 = [6, 17, 19, 34, 90, 149, 83]
tp = [1, 6, 9, 15, 50, 124, 74]
precision, recall, f1, accuracy = compute_weighted_metrics_by_report_bins(N, n, y1, yhat1, tp)
print("Precision: {}, Recall: {}, F1: {}, Acc: {}".format(precision, recall, f1, accuracy))

print("Weights: ", [N[i]/ n[i] for i in range(len(N))])

Precision: 0.5075452992759316, Recall: 0.8609445257418527, F1: 0.6386139509249841, Acc: 0.9265353935113699
Weights:  [31.556701030927837, 20.623376623376622, 21.963730569948186, 13.627986348122867, 4.615384615384615, 2.1577726218097446, 1.2330508474576272]


In [ ]:
# AB (weight by report max threshold 0.4 findings)
N = [3061, 3176, 4239, 3993, 1980, 930, 291]
n = [97, 154, 193, 293, 429, 431, 236]
y1 = [1, 7, 9, 21, 57, 144, 80]
yhat1 = [6, 11, 14, 17, 61, 119, 71]
tp = [1, 4, 6, 12, 40, 105, 67]
precision, recall, f1, accuracy = compute_weighted_metrics_by_report_bins(N, n, y1, yhat1, tp)
print("Precision: {}, Recall: {}, F1: {}, Acc: {}".format(precision, recall, f1, accuracy))

print("Weights: ", [N[i]/ n[i] for i in range(len(N))])

Precision: 0.57118002748295, Recall: 0.6779383834362607, F1: 0.6199970492756024, Acc: 0.937344734279634
Weights:  [31.556701030927837, 20.623376623376622, 21.963730569948186, 13.627986348122867, 4.615384615384615, 2.1577726218097446, 1.2330508474576272]


In [ ]:
# AB (weight by pt max threshold 0.4)

N = [3061, 3176, 4239, 3993, 1980, 930, 291]
# actual n used for test in correct bins
n = [97, 79, 60, 50, 40, 22, 9]
y1 = [1, 5, 4, 5, 7, 9, 3]
yhat1 = [6, 15, 13, 16, 24, 16, 8]
tp = [1, 5, 4, 5, 7, 9, 3]
precision, recall, f1, accuracy = compute_weighted_metrics_by_report_bins(N, n, y1, yhat1, tp)
print("Precision: {}, Recall: {}, F1: {}, Acc: {}".format(precision, recall, f1, accuracy))

print("Weights: ", [N[i]/ n[i] for i in range(len(N))])

Precision: 0.3400926556215047, Recall: 1.0, F1: 0.5075658823960607, Acc: 0.8091004765599269
Weights:  [31.556701030927837, 40.20253164556962, 70.65, 79.86, 49.5, 42.27272727272727, 32.333333333333336]


In [ ]:
# AB (weight by patient avg threshold 0.5)
N = [3061, 3176, 4239, 3993, 1980, 930, 291]
# actual n used for test in correct bins
n = [97, 79, 60, 50, 40, 22, 9]
y1 = [1, 5, 4, 5, 7, 9, 3]
yhat1 = [6, 15, 13, 16, 24, 16, 8]
tp = [1, 5, 4, 5, 7, 9, 3]
precision, recall, f1, accuracy = compute_weighted_metrics_by_report_bins(N, n, y1, yhat1, tp)
print("Precision: {}, Recall: {}, F1: {}, Acc: {}".format(precision, recall, f1, accuracy))

print("Weights: ", [N[i]/ n[i] for i in range(len(N))])

Precision: 0.3400926556215047, Recall: 1.0, F1: 0.5075658823960607, Acc: 0.8091004765599269
Weights:  [31.556701030927837, 40.20253164556962, 70.65, 79.86, 49.5, 42.27272727272727, 32.333333333333336]


In [ ]:
# AB (Incidence rate avg threshold 0.5)
N = [2099, 2044, 2707, 2675, 1223, 659, 162]
# actual n used for test in correct bins
n = [71, 108, 128, 221, 265, 292, 161]
y1 = [2, 5, 8, 22, 67, 131, 80]
yhat1 = [3, 6, 8, 19, 49, 104, 62]
tp = [2, 5, 8, 17, 48, 104, 61]
precision, recall, f1, accuracy = compute_weighted_metrics_by_report_bins(N, n, y1, yhat1, tp)
print("Precision: {}, Recall: {}, F1: {}, Acc: {}".format(precision, recall, f1, accuracy))

print("Weights: ", [N[i]/ n[i] for i in range(len(N))])

Precision: 0.9303615146898073, Recall: 0.8209147373589432, F1: 0.8722181638527594, Acc: 0.9734999693234736
Weights:  [29.56338028169014, 18.925925925925927, 21.1484375, 12.104072398190045, 4.615094339622641, 2.256849315068493, 1.0062111801242235]


In [ ]:
# AB (Incidence rate avg threshold 0.4)
N = [2099, 2044, 2707, 2675, 1223, 659, 162]
# actual n used for test in correct bins
n = [71, 108, 128, 221, 265, 292, 161]
y1 = [2, 5, 8, 22, 67, 131, 80]
yhat1 = [5, 12, 9, 22, 57, 119, 76]
tp = [2, 5, 8, 18, 53, 117, 74]
precision, recall, f1, accuracy = compute_weighted_metrics_by_report_bins(N, n, y1, yhat1, tp)
print("Precision: {}, Recall: {}, F1: {}, Acc: {}".format(precision, recall, f1, accuracy))

print("Weights: ", [N[i]/ n[i] for i in range(len(N))])

Precision: 0.7806948771860438, Recall: 0.8817965848148613, F1: 0.8281715632470853, Acc: 0.9596867822968822
Weights:  [29.56338028169014, 18.925925925925927, 21.1484375, 12.104072398190045, 4.615094339622641, 2.256849315068493, 1.0062111801242235]


In [23]:
# BC
N = [720.4928131, 7.333333333, 4.2, 1, 1, 0, 2, 0, 3, 232.1885714]
n = [172, 6, 3, 1, 1, 0, 2, 0, 3, 227]
y1 = [16, 2, 3, 0, 0, 0, 0, 0, 1, 195]
precision, recall, accuracy = compute_weighted_metrics(N, n, y1, 0.5)
f1  = (2 * precision * recall / (precision + recall)
        if (not math.isnan(precision) and not math.isnan(recall) and (precision + recall) > 0)
        else float('nan'))
print('Precision: ', precision)
print('Recall:', recall)
print('Acc: ', accuracy)
print("F1", 2 * precision * recall / (precision + recall))

Precision:  0.8451382865936816
Recall: 0.7312640089750653
Acc:  0.8863295023749491
F1 0.7840881903433659


In [24]:
N = [720.4928131, 7.333333333, 4.2, 1, 1, 0, 2, 0, 3, 232.1885714]
n = [172, 6, 3, 1, 1, 0, 2, 0, 3, 227]
y1 = [16, 2, 3, 0, 0, 0, 0, 0, 1, 195]
precision, recall, accuracy = compute_weighted_metrics(N, n, y1, 0.4)
f1  = (2 * precision * recall / (precision + recall)
        if (not math.isnan(precision) and not math.isnan(recall) and (precision + recall) > 0)
        else float('nan'))
print('Precision: ', precision)
print('Recall:', recall)
print('Acc: ', accuracy)
print("F1", 2 * precision * recall / (precision + recall))

Precision:  0.8415900966800085
Recall: 0.7312640089750653
Acc:  0.8852998639421304
F1 0.7825577029671411
